In [1]:
import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
import plotly.graph_objects as go

c:\Users\dell\Desktop\llmEngineering\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

True

In [3]:
knowledge_base_path = "knowledge-base/**/*.md"
files = glob.glob(knowledge_base_path, recursive=True)
print(f"Found {len(files)} files in the knowledge base")

entire_knowledge_base = ""

for file_path in files:
    with open(file_path, 'r', encoding='utf-8') as f:
        entire_knowledge_base += f.read()
        entire_knowledge_base += "\n\n"

print(f"Total characters in knowledge base: {len(entire_knowledge_base):,}")

Found 76 files in the knowledge base
Total characters in knowledge base: 304,434


In [4]:
#  How many tokens in all the documents?
MODEL = "gpt-4.1-nano"
tokenModel = tiktoken.encoding_for_model(MODEL)
tokens = tokenModel.encode(entire_knowledge_base)
tokens

[2,
 12958,
 10608,
 627,
 680,
 76,
 279,
 13345,
 627,
 680,
 76,
 673,
 24303,
 656,
 135158,
 100802,
 306,
 220,
 667,
 20,
 472,
 448,
 8112,
 6705,
 34220,
 6884,
 316,
 44258,
 448,
 5735,
 306,
 1309,
 328,
 18186,
 3891,
 13,
 13974,
 1577,
 1888,
 673,
 5070,
 596,
 76,
 11,
 290,
 33620,
 29430,
 16398,
 483,
 8112,
 15230,
 364,
 976,
 3175,
 11790,
 12379,
 8266,
 306,
 1617,
 1577,
 6468,
 2101,
 11,
 35479,
 1617,
 1888,
 22554,
 316,
 3931,
 4004,
 680,
 76,
 350,
 7254,
 8112,
 26883,
 936,
 17482,
 596,
 76,
 350,
 9603,
 8112,
 26883,
 936,
 326,
 460,
 596,
 76,
 350,
 143677,
 322,
 113676,
 6361,
 741,
 4534,
 220,
 1323,
 15,
 11,
 10608,
 627,
 680,
 76,
 1458,
 15237,
 261,
 25689,
 328,
 220,
 1179,
 10789,
 483,
 220,
 899,
 25470,
 5251,
 290,
 2952,
 364,
 18683,
 11,
 290,
 3175,
 93968,
 261,
 23973,
 116848,
 306,
 220,
 1323,
 17,
 12,
 1323,
 18,
 316,
 5184,
 402,
 82382,
 326,
 24411,
 8266,
 13,
 1328,
 7360,
 30749,
 1365,
 6885,
 14245,
 11,
 368

In [5]:
folders = glob.glob("knowledge-base/*")
documents = []
for folder in folders:
    doc_type = os.path.basename(folder)

    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"Loaded {len(documents)} documents")

Loaded 76 documents


In [6]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
chunk = text_splitter.split_documents(documents=documents)

In [7]:
len(chunk)

413

In [8]:
#  Using huggingfacel embedding model
db_name = "vector_db"

embeddings = HuggingFaceEmbeddings(model_name = "all-MiniLM-L6-v2")
# embeddings = OpenAIEmbeddings(model = "text-embedding-3-small")

if os.path.exists(db_name):
    Chroma(persist_directory=db_name,embedding_function=embeddings).delete_collection()
vectorstore = Chroma.from_documents(documents=chunk, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3736.69it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vectorstore created with 413 documents


In [9]:
# Let's investigate the vectors

collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

There are 413 vectors with 384 dimensions in the vector store


In [10]:
result = collection.get(include=['embeddings', 'documents', 'metadatas'])
result

{'ids': ['b020c4f0-a56b-4720-a3a8-80ed32e4c26a',
  'dae542e1-b45f-4ca4-b7e6-0c8a30551055',
  'cad85f6e-0c22-4ff6-bbdc-112f2398f508',
  '6a817069-05af-4155-969a-be75f9e1fbd6',
  '63459091-3da6-49a0-908a-ebe43739c652',
  '0e386a0a-18de-4667-bc18-63a46a8ed19b',
  '86de6ea4-bc12-45a9-98b1-5d3434118221',
  '025de787-7015-4afb-a865-28ad2e604dd4',
  'cfa70a54-0f28-4429-b8a5-7dea3752a888',
  '5af6e93e-3b93-44b5-829f-4168c81d7274',
  'f427f34f-d287-4ec7-bac2-2097af7f16b9',
  '9406d799-4a55-4f97-be75-259f912b27d6',
  '33c9d67a-a82a-4286-8287-59e1e1852638',
  'a84b57f7-f956-4a1c-a9c6-60f101ba5f7a',
  'eaf1cef9-7c30-4aca-b73c-d36d9869ca0e',
  '99c5c0e8-5278-431d-a11f-070fd4dedc25',
  '1e2023c9-1097-4333-999f-654e579cec8b',
  'e0cf92d7-ac7c-4779-8bdb-75fc938565f2',
  '8e733ed5-536a-486e-bfc6-200ca6d8951e',
  '6ebee813-1b1e-4e14-aa62-419ef91041d6',
  'd24f80fe-c607-4e6c-8498-f5aebd47710c',
  'e66ccae5-c21b-410c-b73d-70be82b73879',
  'f1def554-d302-4419-942c-8604e5cba910',
  '3c2c4396-f32d-440c-b8a6-